In [32]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [33]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"


In [34]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"


### La table de travail


In [35]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)


### La cible, et la référence


In [36]:
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())


In [37]:
y = train[CIBLE]
reference = 100 * max(y.mean(), 1 - y.mean())


### Le premier modèle


In [38]:
# le modèle et le découpage : ils entrent ici
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
# Conserve la reference avant que remesurer() ne remplace modele pour la question 10.
modele_reference = modele
score_interne = 100 * modele.score(Xval, yval)


### Vos features — LA cellule que vous modifiez


In [39]:
# votre feature : elle reçoit la TABLE DE TRAVAIL — identifiants, dates et colonnes post-issue
# déjà retirés — et la rend enrichie. `predict` l'appelle sur la table BRUTE, après avoir retiré
# les mêmes colonnes : le crochet voit donc la même forme des deux côtés, ce qui est la règle.
def ajouter_features(df):
    """Signale les ages codes a -1, valeur impossible observee a la question 3."""
    df = df.copy()
    df["age_sentinelle"] = (pd.to_numeric(df["age"], errors="coerce") == -1).astype("int8")
    return df
# Votre feature entre par `preparer()`, la chaîne unique : on n'applique RIEN à la main.
# Après l'avoir écrite, appelez `remesurer()` (au devoir, dans la cellule prévue) : `X` la
# reçoit une seule fois, et `predict` l'applique aussi, de son côté, sur la table brute.


### La chaîne — une seule, des deux côtés


In [40]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.3, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### L'interface — le modèle qu'on peut appeler


In [41]:
# numpy, pour la seule assertion qui suit
import numpy as np

def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours appellera.
       Elle passe par `preparer(...)`, la même chaîne que l'entraînement — pas une copie."""
    # elle ne refait PAS la chaîne à la main : elle appelle `preparer`, celle de l'entraînement.
    # C'est ce couplage qui garantit qu'une table n'est jamais préparée de deux façons.
    return modele.predict(preparer(df)).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"
    print(f"test OK — 500 prédictions, {int(sum(p))} positifs")

# test_predict()   (au devoir : la cellule « l'interface » s'en charge)


### Le tableau d'essais


In [42]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

# les deux amorces du carnet sont annoncées elles aussi — sinon elles ne passeraient pas
prevoir("rien faire du tout", "c'est le minimum à battre",
        "un modèle qui répond toujours la même chose vaut la classe majoritaire")
prevoir("un modèle suffit-il ?", "mieux que la référence",
        "les arbres captent une partie du signal, pas tout")


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:45:56


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:45:56
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:45:56


### La soumission


In [43]:
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)


### Le modèle linéaire


In [44]:
# le modèle linéaire : il entre ici
from sklearn.linear_model import LogisticRegression
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]

modele_lineaire = LogisticRegression(solver="newton-cholesky").fit(Xtr_n, ytr)
score_lineaire = 100 * modele_lineaire.score(Xval_n, yval)


### Les outils du devoir : les tableaux, la barre, les contrôles


In [45]:
# les trois tableaux, et les outils qui les remplissent

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (un nombre) · conséquence : garder, écarter,
       surveiller, mesurer. Ré-exécuter une cellule REMPLACE la ligne de même `observation`."""
    ligne = {"observation": observation, "chiffre": chiffre,
             "comparaison": comparaison, "conséquence": consequence}
    CONSTATS[:] = [c for c in CONSTATS if c["observation"] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT.
       Ré-exécuter une cellule REMPLACE la ligne de même `feature`."""
    ligne = {"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
             "test": comment_on_la_test, "contredite si": ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h["feature"] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

# les trois constats de la séance 1 — ils comptent : la question 3 en demande CINQ au total,
# donc DEUX des vôtres. Ils sont réécrits ici parce que ce carnet doit tourner seul.
constat("une colonne ne dit rien",
        f"{train['zone_clinique'].nunique()} valeur(s) distincte(s)",
        f"sur les {len(train)} lignes de la table, soit une colonne constante au sens du cours",
        "l'écarter — et ce qu'on y gagne se mesure")
constat("deux colonnes disent la même chose",
        "corrélation 1,000 entre cout_acte et montant_acte",
        f"contre moins de 0,05 pour les autres paires sur {len(train)} lignes : un rapport constant",
        "n'en garder qu'une")
constat("la cible est déséquilibrée",
        f"{100 * train[CIBLE].mean():.2f} % d'absents",
        "contre 50 % pour un problème équilibré : la référence monte d'autant",
        "mesurer : cela conditionne la métrique, donc ce qu'on regarde")
CONSTATS_DE_LA_SEANCE = [c['observation'] for c in CONSTATS]

# les deux amorces de J1 — annoncées, puis notées (c'est la règle de l'ordre, montrée)
noter("rien faire du tout", "classe majoritaire", "c'est le minimum à battre", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)

# la barre de bruit — recalculée ici parce que la question 5 en change la taille du jeu
erreur_type = 100 * ((score_interne / 100) * (1 - score_interne / 100) / len(yval)) ** 0.5
print(f"erreur-type de l'exactitude de validation (jeu de 30 %) : ± {erreur_type:.2f} point(s)")
print("  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.")
print("  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.")

# ── VOTRE échantillon : il dépend de VOTRE identifiant GitHub, et de lui seul.
#    Les questions 2 et 11 mesurent dessus : deux voisins n'ont donc pas les mêmes chiffres,
#    et un chiffre recopié ne tombe pas juste. Le reste du devoir (le socle J1/J2) garde le
#    protocole figé du cours — c'est voulu : ce qui doit être comparable entre tous reste
#    comparable, et ce qui doit être VOTRE mesure est la vôtre.
IDENTIFIANT_GITHUB = "Shiro26-rg"          # <- remplacez par votre pseudo GitHub, comme dans l'URL
import hashlib
if IDENTIFIANT_GITHUB.strip() and IDENTIFIANT_GITHUB.strip() != '…':
    GRAINE = int(hashlib.sha256(IDENTIFIANT_GITHUB.strip().lower().encode('utf-8')).hexdigest()[:8], 16)
else:
    GRAINE = None
mon_train = train.sample(frac=0.6, random_state=GRAINE) if GRAINE is not None else train
print(f"votre échantillon : {len(mon_train)} lignes sur {len(train)}"
      + ("" if GRAINE is not None else "   (identifiant non renseigné : la table entière)"))

# la table de travail NUE, gardée de côté : c'est sur elle que `remesurer()` rebâtit `X`,
# pour que votre feature s'applique UNE SEULE fois, même si vous relancez la cellule dix fois.
X_sans_features = matrice(train)[0]

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour — et `X` avec.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis la table nue : ré-exécutable."""
    global modele, X
    X = ajouter_features(X_sans_features)
    a, b, c, d = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)

VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit JAMAIS la valeur attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    if condition:
        print('  ok        ' + nom)
    else:
        print('  À REVOIR  ' + nom + '   ->   ' + indice)

def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de FORME, posé dans la cellule de la question. Il ne compare JAMAIS à une valeur
       attendue : un type, une plage, une dimension, ou une cohérence avec VOS autres réponses.
       Tant que la réponse n'est pas écrite, il se tait.

       `des_que` : une fonction sans argument qui dit si la réponse EXISTE déjà. Tant qu'elle rend
       False, le contrôle se tait — un contrôle parle d'une réponse écrite, jamais de son absence."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))

# le nombre de colonnes de la table AVANT votre feature (la question 10 s'y compare)
n_colonnes_avant = X.shape[1]

# les deux essais du haut du carnet J1 sont des AMORCES : ils ne comptent pas comme les vôtres.
AMORCES = ('rien faire du tout', 'un modèle suffit-il ?')

# la clé de la question 15 vit côté enseignant, pas ici : un vocabulaire de quatre familles,
# c'est 4^6 combinaisons, et une empreinte de nom de colonne se casse en quelques secondes.


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 19:45:56"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 19:45:56"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.49,"oui, à 19:45:56"


erreur-type de l'exactitude de validation (jeu de 30 %) : ± 0.18 point(s)
  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.
  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.
votre échantillon : 89936 lignes sur 149894


### L'interface, telle qu'on l'appelle


In [46]:
# `predict` est définie par la mise en place, mais elle est testée ICI : ainsi, `Run All`
# fonctionne même après que vous avez écrit votre feature (la Q9 de l'ancien devoir cassait là).
# Relancez CETTE cellule après avoir touché `ajouter_features` : `X` reçoit la feature une seule
# fois, `modele` est mis à jour, et `predict` l'applique aussi de son côté.
print(f"exactitude de validation, table de travail courante : {remesurer():.2f} %")
test_predict()

import matplotlib.pyplot as plt          # pour les graphiques des questions 2 et 13


exactitude de validation, table de travail courante : 82.49 %
test OK — 500 prédictions, 28 positifs


## Les questions

**Quinze questions : douze de cœur, trois d'approfondissement.** Les huit premières portent sur
**J1**, les sept dernières sur **J2**. Pour chacune : lisez la consigne, écrivez votre code (ou
votre phrase) à l'emplacement prévu, et **exécutez la cellule** — c'est l'exécution qui déclenche
les contrôles.


### Question 1 — La référence, recalculée et prouvée — et le collègue qui annonce 83 %  ·  *carnet J1*  ·  *cœur*


La référence est le chiffre qui décide si tout ce qui suit vaut quelque chose : le score
d'un modèle qui répond **toujours la même chose**. Le carnet l'a calculée sur **tout** `train`.

À faire :
  1. calculez la part d'absents **sur les lignes de contrôle** (`yval`), en **une** ligne, sans
     réutiliser `reference` → `part_absents_controle` ;
  2. **deux chiffres différents, et c'est le piège de la séance** : cette part d'absents n'est **pas**
     la référence. La référence est une **exactitude** — la part de lignes bien classées par un
     modèle qui répond toujours la classe majoritaire (ici : « le patient vient ») : calculez-la
     **sur les lignes de contrôle** → `ma_reference` ;
  3. construisez ce prédicteur : un **tableau de 0/1 de la même longueur que `yval`**, rempli de la
     classe majoritaire — puis montrez qu'il **atteint** `ma_reference` — pas « environ » ;
  4. écrivez l'écart entre `ma_reference` et `reference` (celle de la mise en place, calculée sur
     tout `train`) → `ecart_a_train`, et **pourquoi ce petit écart compte** quand on compare deux
     modèles ;
  5. **un collègue annonce 83 %** d'exactitude. Écrivez votre réponse en **deux chiffres** :
     l'écart à la référence, puis **le nombre de barres de bruit** que vaut cet écart. Un
     score qui ne bat pas la référence de deux barres n'est pas un résultat.

À écrire : `part_absents_controle`, `ma_reference`, `ecart_a_train`, `reponse_au_collegue`,
`ecart_en_barres`.


In [47]:
# Part des absents et reference d'exactitude de la classe majoritaire, sur yval.
part_absents_controle = 100 * yval.mean()
ma_reference = 100 * max(yval.mean(), 1 - yval.mean())
pred = np.full(len(yval), int(yval.mean() >= 0.5), dtype=int)
assert abs(100 * (pred == yval).mean() - ma_reference) < 1e-10

ecart_a_train = ma_reference - reference
ecart_collegue = 83 - ma_reference
ecart_en_barres = ecart_collegue / erreur_type
reponse_au_collegue = (
    f"83 % est {ecart_collegue:+.2f} points de la reference ({ma_reference:.2f} %), "
    f"soit {ecart_en_barres:+.2f} barre(s) de bruit. "
    + ("Il ne depasse donc pas la reference de deux barres : ce n'est pas un resultat convaincant. "
       if ecart_en_barres < 2 else
       "Il depasse la reference de plus de deux barres : le gain est plus convaincant, "
       "sans prouver a lui seul une generalisation. ")
    + f"L'ecart de {ecart_a_train:+.2f} point(s) entre les deux references vient du tirage de validation; "
      "il rappelle qu'une petite difference entre modeles peut venir du decoupage."
)
controle('la part d absents est un pourcentage, et c est la classe MINORITAIRE',
         lambda: 0 < part_absents_controle < 50,
         'la classe minoritaire : c est elle qu on cherche à détecter')
controle('`ma_reference` est un pourcentage, au-dessus de 50',
         lambda: 50 < ma_reference <= 100,
         'la classe majoritaire fait plus de la moitié : la référence est son score')
controle('les deux chiffres sont complémentaires',
         lambda: abs((part_absents_controle + ma_reference) - 100) < 0.05,
         'part d absents + référence = 100 % : deux faces du même tirage')
controle('le prédicteur ne répond que des 0 et des 1', lambda: set(pred) <= {0, 1})
controle('son score COÏNCIDE avec votre référence',
         lambda: abs((pred == yval).mean() * 100 - ma_reference) < 0.05,
         'les deux mesures doivent tomber ensemble : c est le point de la question')
controle('l écart à `reference` est chiffré', lambda: abs(ecart_a_train) < 5,
         'deux chiffres proches, séparés par un écart de tirage')
controle('la réponse au collègue porte deux chiffres',
         lambda: len(reponse_au_collegue) > 60 and 0 < abs(ecart_en_barres) < 30,
         'écrivez l écart à la référence ET le nombre de barres')


  ok        la part d absents est un pourcentage, et c est la classe MINORITAIRE
  ok        `ma_reference` est un pourcentage, au-dessus de 50
  ok        les deux chiffres sont complémentaires
  ok        le prédicteur ne répond que des 0 et des 1
  ok        son score COÏNCIDE avec votre référence
  ok        l écart à `reference` est chiffré
  ok        la réponse au collègue porte deux chiffres


### Question 2 — Le balayage, le piège des valeurs rares, et le critère des 500 lignes  ·  *carnet J1*  ·  *cœur*


Trouvez, en regroupant, la colonne qui **sépare le mieux** les deux réponses. Un premier
balayage, **sans aucun filtre**, va désigner une colonne très probablement inutile.

**Tout se fait sur `mon_train` — VOTRE échantillon**, celui que la mise en place a tiré à partir
de votre identifiant GitHub : vos chiffres ne sont pas ceux de votre voisin.

À faire :
  1. pour **toutes** les colonnes de `mon_train`, groupez **par cette colonne** et prenez la
     **moyenne de la cible** dans chaque groupe : c'est le **taux d'absence** du groupe, la colonne
     `taux_absence` de la séance 1. L'écart cherché est celui entre le taux le plus haut et le taux
     le plus bas — sans filtre, et triez ;
  2. notez **laquelle gagne** et son écart : c'est `colonne_piegee` et `ecart_non_filtre`. Comptez
     combien de ses groupes atteignent 500 lignes : c'est `groupes_de_500` ;
  3. **puis** appliquez le critère du cours : **ne gardez que les groupes d'au moins 500 lignes, et
     comparez ceux-là** — refaites le classement (`classement`), et notez le plus petit groupe gardé
     (`plus_petit_groupe_du_classement`) ;
  4. `meilleure_colonne` et son écart `ecart_filtre` ; puis `ce_que_le_critere_ne_protege_pas` : le
     critère a-t-il suffi à écarter le piège ? Regardez ce qui monte en tête du classement, et
     demandez-vous **ce qu'il faudrait vérifier en plus** du critère ;
  5. `lecture` : pourquoi la colonne du premier balayage gagnait-elle ? par rapport à quoi ?

À écrire : `balayage_brut`, `colonne_piegee`, `ecart_non_filtre`, `groupes_de_500`, `classement`,
`plus_petit_groupe_du_classement`, `meilleure_colonne`, `ecart_filtre`,
`ce_que_le_critere_ne_protege_pas`, `lecture`.


In [48]:
# On balaie tous les predicteurs de l'echantillon; la cible elle-meme n'est pas un predicteur.
import pandas as pd
lignes_brutes = []
for colonne in (c for c in mon_train.columns if c != CIBLE):
    groupes = mon_train.groupby(colonne, dropna=False, observed=True)[CIBLE].agg(["mean", "size"])
    lignes_brutes.append({
        "colonne": colonne,
        "ecart": 100 * (groupes["mean"].max() - groupes["mean"].min()),
        "groupes_de_500": int((groupes["size"] >= 500).sum()),
        "priorite_id": int(colonne == ID),
    })
balayage_brut = pd.DataFrame(lignes_brutes).sort_values(
    ["ecart", "priorite_id"], ascending=[False, False]
).set_index("colonne")
colonne_piegee = balayage_brut.index[0]
ecart_non_filtre = float(balayage_brut.iloc[0]["ecart"])
groupes_de_500 = int(balayage_brut.iloc[0]["groupes_de_500"])

# Le seuil est applique avant de recalculer l'ecart entre taux de groupes conserves.
lignes_filtrees = []
for colonne in mon_train.columns:
    if colonne == CIBLE:
        continue
    groupes = mon_train.groupby(colonne, dropna=False, observed=True)[CIBLE].agg(["mean", "size"])
    conserves = groupes.loc[groupes["size"] >= 500]
    if len(conserves) >= 2:
        lignes_filtrees.append({
            "colonne": colonne,
            "ecart_filtre": 100 * (conserves["mean"].max() - conserves["mean"].min()),
            "plus_petit_groupe": int(conserves["size"].min()),
            "groupes_conserves": len(conserves),
        })
if not lignes_filtrees:
    raise ValueError("Aucune colonne n'a au moins deux groupes de 500 lignes.")
classement = pd.DataFrame(lignes_filtrees).sort_values("ecart_filtre", ascending=False).set_index("colonne")
meilleure_colonne = classement.index[0]
ecart_filtre = float(classement.iloc[0]["ecart_filtre"])
plus_petit_groupe_du_classement = int(classement["plus_petit_groupe"].min())
ce_que_le_critere_ne_protege_pas = (
    f"Le filtre laisse {meilleure_colonne} en tete (ecart {ecart_filtre:.2f} points). "
    "500 lignes limitent les groupes trop rares, mais ne corrigent ni fuite d'information, "
    "ni confusion, ni recherche parmi de nombreuses colonnes; il faut verifier le sens de la variable "
    "et valider sur un decoupage temporel ou independant."
)
lecture = (
    f"Sans filtre, {colonne_piegee} gagne avec {ecart_non_filtre:.2f} points car les groupes minuscules "
    "peuvent avoir des taux 0 ou 100 % par hasard; l'ecart compare le maximum au minimum, "
    "pas un gain moyen ni une preuve causale."
)
display(balayage_brut.head(10))
display(classement.head(10))
controle('le balayage couvre toutes les colonnes', lambda: len(balayage_brut) >= 30,
         'une ligne par colonne de `train`, valeurs d identifiant comprises')
controle('l écart non filtré est de l ordre de 100 points',
         lambda: 50 < ecart_non_filtre <= 100, 'le piège donne un écart énorme')
controle('la colonne piégée n a AUCUN groupe de 500 lignes',
         lambda: groupes_de_500 == 0,
         'chaque ligne a son propre groupe : ce n est pas une trouvaille')
controle('le classement filtré est plus petit que le non filtré',
         lambda: len(classement) < len(balayage_brut),
         'le critère des 500 lignes retire des colonnes')
controle('la meilleure colonne est bien celle de VOTRE classement',
         lambda: meilleure_colonne == classement.index[0],
         'la première de votre propre classement — l écart est net, pas d hésitation')
controle('le classement filtré ne garde QUE des groupes de 500 lignes et plus',
         lambda: plus_petit_groupe_du_classement >= 500,
         'le critère se pose AVANT la comparaison : ne gardez que les gros groupes')
controle('les deux écarts sont chiffrés en POINTS, et le filtré est plus petit',
         lambda: 0 < ecart_filtre <= 100 and ecart_filtre < ecart_non_filtre,
         'en points (0 à 100), pas en proportion')
controle('la lecture compare, et le critère est interrogé',
         lambda: len(lecture) > 60 and len(ce_que_le_critere_ne_protege_pas) > 60,
         'par rapport à quoi ? et le critère des 500 lignes suffit-il ?')


,ecart,groupes_de_500,priorite_id
colonne,,,
ligne_id,100.0,0,1
dossier_id,100.0,0,0
patient_id,100.0,0,0
motif_absence,100.0,5,0
taille_cm,100.0,0,0
date_prise,100.0,0,0
encours_1_mois,100.0,0,0
encours_3_mois,100.0,0,0
encours_6_mois,100.0,0,0


,ecart_filtre,plus_petit_groupe,groupes_conserves
colonne,,,
motif_absence,100.000000,2979,5
duree_consultation,99.417864,534,30
canal_prise,42.290422,11518,4
taux_absence_passe,29.217295,923,3
delai_jours,25.756152,4783,7
constantes_relevees,24.096308,10399,3
quartier,21.421545,527,21
clinique,20.736533,903,80
rappel_envoye,15.190979,30284,2


  ok        le balayage couvre toutes les colonnes
  ok        l écart non filtré est de l ordre de 100 points
  ok        la colonne piégée n a AUCUN groupe de 500 lignes
  ok        le classement filtré est plus petit que le non filtré
  ok        la meilleure colonne est bien celle de VOTRE classement
  ok        le classement filtré ne garde QUE des groupes de 500 lignes et plus
  REVOIR    les deux écarts sont chiffrés en POINTS, et le filtré est plus petit   ->   en points (0 à 100), pas en proportion
  ok        la lecture compare, et le critère est interrogé


### Question 3 — Cinq constats chiffrés — trois de la séance, deux de vous  ·  *carnet J1*  ·  *cœur*


Cinq constats, **dont au moins deux mesurés par vous**. Les trois premiers sont ceux de la
séance 1 : ils sont déjà écrits par la mise en place, ne les réécrivez pas.

À faire : ajoutez **deux** constats avec `constat(...)` — trois de la séance plus deux des vôtres,
cinq au total — et écrivez-les pour qu'un programme puisse les vérifier, c'est ce qu'il fait :
  · `chiffre` et `comparaison` doivent contenir **un nombre** (un taux, un effectif, un écart) ;
  · `conséquence` doit venir du vocabulaire du cours : **garder**, **écarter**, **surveiller**,
    **mesurer** ;
  · au moins un de vos constats doit être **négatif** — un défaut cherché et non trouvé, ou un
    écart nul. C'est un résultat, pas un échec ;
  · au moins un doit **changer une décision** (garder ou écarter une colonne).

À écrire : les constats, via `constat(...)`.


In [49]:
# Deux constats personnels : un resultat negatif et un defaut qui change une decision.
import re

cibles_manquantes = int(train[CIBLE].isna().sum())
constat(
    "aucune cible manquante" if cibles_manquantes == 0 else "la cible a des valeurs manquantes",
    f"{cibles_manquantes} valeur(s) manquante(s)",
    f"sur {len(train)} lignes de train",
    "garder la cible telle quelle" if cibles_manquantes == 0 else "surveiller les lignes incompletes",
)
valeurs_age = pd.to_numeric(train["age"], errors="coerce")
ages_impossibles = int((valeurs_age < 0).sum())
constat(
    "des ages impossibles sont presents",
    f"{ages_impossibles} age(s) negatif(s)",
    f"sur {len(train)} lignes; zero age negatif est plausible",
    "ecarter la sentinelle negative ou la recoder en valeur manquante avant l'apprentissage",
)
controle('cinq constats au total', lambda: len(CONSTATS) == 5,
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE),
         )
controle('au moins deux constats sont les VÔTRES',
         lambda: sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres',
         des_que=lambda: len(CONSTATS) > len(CONSTATS_DE_LA_SEANCE))
controle('chaque `chiffre` porte un nombre',
         lambda: all(re.search(r'\d', str(c['chiffre'])) for c in CONSTATS),
         'une mesure exacte : un taux, un effectif, un écart')
controle('chaque `comparaison` porte un nombre',
         lambda: all(re.search(r'\d', str(c['comparaison'])) for c in CONSTATS),
         'par rapport à QUOI : un nombre, pas une impression')
controle('chaque `conséquence` vient du vocabulaire du cours',
         lambda: all(any(m in str(c['conséquence']).lower()
                         for m in ('garder', 'ecarter', 'écarter', 'surveiller', 'mesurer'))
                     for c in CONSTATS),
         'garder, écarter, surveiller ou mesurer — pas autre chose')
controle('au moins un constat négatif',
         lambda: any(m in ' '.join(str(c['observation']) + ' ' + str(c['chiffre'])
                                   for c in CONSTATS).lower()
                     for m in ('aucun', 'aucune', 'rien', 'zéro', 'zero', '0 ', 'nul')),
         'un défaut cherché et non trouvé est un résultat : écrivez-le')
controle('au moins un constat change une décision',
         lambda: any(('garder' in str(c['conséquence']).lower()
                      or 'ecarter' in str(c['conséquence']).lower()
                      or 'écarter' in str(c['conséquence']).lower()) for c in CONSTATS),
         'au moins un constat doit dire ce qu on garde ou ce qu on écarte')


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucune cible manquante,0 valeur(s) manquante(s),sur 149894 lignes de train,garder la cible telle quelle


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucune cible manquante,0 valeur(s) manquante(s),sur 149894 lignes de train,garder la cible telle quelle
4,des ages impossibles sont presents,46 age(s) negatif(s),sur 149894 lignes; zero age negatif est plausible,ecarter la sentinelle negative ou la recoder e...


  ok        cinq constats au total
  ok        au moins deux constats sont les VÔTRES
  ok        chaque `chiffre` porte un nombre
  ok        chaque `comparaison` porte un nombre
  ok        chaque `conséquence` vient du vocabulaire du cours
  ok        au moins un constat négatif
  ok        au moins un constat change une décision


### Question 4 — Trois défauts établis — une prédiction chiffrée, puis la même sur le modèle linéaire  ·  *carnet J1*  ·  *cœur*


Trois défauts sont **établis** : un nombre rangé en texte, des lignes dupliquées, un âge
impossible. Un diagnostic **certain** ne dit rien de ce qu'il **coûte**.

À faire :
  1. **AVANT de mesurer** : pour chacun des trois, écrivez ce que vous attendez — le score
     bouge-t-il, et **de combien de barres de bruit** ? (`pred_age`, `pred_doublons`,
     `pred_taille`, en barres) ;
  2. mesurez la correction de l'âge sur l'ensemble d'arbres : `s_avec_age`, `s_sans_age`, et
     l'écart en barres (`age_en_barres`) ;
  3. **faites la MÊME correction sur le modèle linéaire** (`modele_lineaire` est dans la mise
     en place) : `lin_avec_age`, `lin_sans_age`, `lin_age_en_barres`. Dites ce que devient un
     âge à −1 pour un arbre, et ce qu'il devient pour une somme pondérée ;
  4. `verdict_age` : au vu de **vos deux mesures**, ce défaut compte-t-il — et pour qui ?
     Écrivez votre réponse, et ce qui la distinguerait du verdict qu'appellerait un défaut d'un
     autre effet. Puis `verdict_doublons` : que vaut un contrôle qui ne trouve **rien** ?

À écrire : les trois prédictions en barres, les deux mesures, les deux verdicts.


In [50]:
# Predictions AVANT les mesures : j'attends peu d'effet des trois corrections sur l'exactitude.
# Un arbre peut isoler une sentinelle comme -1; des doublons et des valeurs extremes peuvent toutefois
# modifier les seuils, d'ou une prediction proche de zero plutot qu'une certitude.
pred_age = 0.0
pred_doublons = 0.0
pred_taille = 0.0

# Meme decoupage pour comparer l'inclusion/exclusion de age, sur arbres puis sur modele lineaire.
colonnes_avec_age = list(Xtr.columns)
colonnes_sans_age = [c for c in Xtr.columns if c != "age"]
modele_avec_age = HistGradientBoostingClassifier(random_state=0).fit(Xtr[colonnes_avec_age], ytr)
modele_sans_age = HistGradientBoostingClassifier(random_state=0).fit(Xtr[colonnes_sans_age], ytr)
s_avec_age = 100 * modele_avec_age.score(Xval[colonnes_avec_age], yval)
s_sans_age = 100 * modele_sans_age.score(Xval[colonnes_sans_age], yval)
age_en_barres = (s_sans_age - s_avec_age) / erreur_type

Xtr_lineaire = X_num.loc[Xtr.index]
Xval_lineaire = X_num.loc[Xval.index]
modele_lineaire_avec_age = LogisticRegression(solver="newton-cholesky").fit(Xtr_lineaire, ytr)
colonnes_lineaires_sans_age = [c for c in Xtr_lineaire.columns if c != "age"]
modele_lineaire_sans_age = LogisticRegression(solver="newton-cholesky").fit(
    Xtr_lineaire[colonnes_lineaires_sans_age], ytr
)
lin_avec_age = 100 * modele_lineaire_avec_age.score(Xval_lineaire, yval)
lin_sans_age = 100 * modele_lineaire_sans_age.score(
    Xval_lineaire[colonnes_lineaires_sans_age], yval
)
lin_age_en_barres = (lin_sans_age - lin_avec_age) / erreur_type

verdict_age = (
    f"Retirer age change le score de {age_en_barres:+.2f} barre(s) pour les arbres et "
    f"{lin_age_en_barres:+.2f} pour le lineaire. Un arbre peut isoler la valeur -1 par un seuil; "
    "dans une somme ponderee, -1 est multiplie par le coefficient comme s'il s'agissait d'un age reel. "
    "Le defaut compte pour le modele dont l'ecart depasse deux barres; les deux scores permettent de le juger."
)
verdict_doublons = (
    "Un controle sans doublon exact ne prouve pas qu'il n'existe aucune repetition pertinente : "
    "il peut manquer des doublons avec une seule valeur differente ou des doublons metier. "
    "Ici, il faut comparer la correction sur le meme decoupage et ne conclure a un effet que si le gain "
    "depasse le bruit, pas sur le seul diagnostic de doublons."
)
controle('les trois prédictions sont écrites avant toute mesure',
         lambda: abs(pred_age) < 20 and abs(pred_doublons) < 20 and abs(pred_taille) < 20,
         'trois nombres, en barres de bruit')
controle('deux scores d arbres, en pourcentage',
         lambda: 50 < s_avec_age <= 100 and 50 < s_sans_age <= 100)
controle('deux scores linéaires, en pourcentage',
         lambda: 50 < lin_avec_age <= 100 and 50 < lin_sans_age <= 100)
controle('l écart des arbres et celui du linéaire sont chiffrés',
         lambda: abs(age_en_barres) < 10 and abs(lin_age_en_barres) < 10,
         'deux écarts, chacun divisé par la barre')
controle('les deux verdicts sont argumentés',
         lambda: len(verdict_age) > 60 and len(verdict_doublons) > 40,
         'dites pour qui le défaut compte, mesures à l appui')


  ok        les trois prédictions sont écrites avant toute mesure
  ok        deux scores d arbres, en pourcentage
  ok        deux scores linéaires, en pourcentage
  ok        l écart des arbres et celui du linéaire sont chiffrés
  ok        les deux verdicts sont argumentés


### Question 5 — La barre décroît comme 1/racine(n) — vos propres tirages  ·  *carnet J1*  ·  *cœur*


L'entraînement ne change pas. On ne change que la **taille du jeu de contrôle**, et on
regarde de combien le score bouge **tout seul**.

À faire :
  1. gardez le modèle de la mise en place (`modele`), et le jeu de contrôle `Xval`/`yval` ;
  2. tirez **20 fois** 1 000 lignes de `Xval`, mesurez chaque fois (`ecarts_1000`) ; refaites-le
     pour 5 000 lignes (`ecarts_5000`) ;
  3. calculez l'**écart-type** de vos 20 scores pour chaque taille (`ecart_type_1000`,
     `ecart_type_5000`) ;
  4. calculez la barre **annoncée par la formule du cours**, racine de p(1-p)/n, pour chaque
     taille (`formule_1000`, `formule_5000`), et le **rapport** observé/formule
     (`rapport_1000`, `rapport_5000`) ;
  5. `lecture` : dans quel sens la barre bouge quand `n` est multiplié par 5 ? Le rapport
     vaut-il à peu près 1 ? Et **vos 20 tirages suffisent-ils** pour estimer un écart-type ?

À écrire : les deux listes de 20 scores, les deux écarts-types, les deux formules, les deux
rapports, et la lecture.


In [51]:
# Chaque tirage est un sous-echantillon distinct de la validation; on reutilise le modele deja ajuste.
def scores_tires(n, graine):
    if n > len(Xval):
        raise ValueError(f"Il faut {n} lignes, mais Xval n'en contient que {len(Xval)}.")
    scores = []
    for i in range(20):
        x_tire = Xval.sample(n=n, random_state=graine + i)
        y_tire = yval.loc[x_tire.index]
        scores.append(100 * modele_reference.score(x_tire, y_tire))
    return scores

ecarts_1000 = scores_tires(1000, 11000)
ecarts_5000 = scores_tires(5000, 15000)
ecart_type_1000 = float(np.std(ecarts_1000, ddof=1))
ecart_type_5000 = float(np.std(ecarts_5000, ddof=1))
p_validation = score_interne / 100
formule_1000 = 100 * np.sqrt(p_validation * (1 - p_validation) / 1000)
formule_5000 = 100 * np.sqrt(p_validation * (1 - p_validation) / 5000)
rapport_1000 = ecart_type_1000 / formule_1000
rapport_5000 = ecart_type_5000 / formule_5000
lecture = (
    f"Quand n est multiplie par 5, la formule prevoit une barre divisee par racine(5) "
    f"({formule_1000:.3f} a {formule_5000:.3f} points). Les rapports observes/formule sont "
    f"{rapport_1000:.2f} et {rapport_5000:.2f}; ils se jugent autour de 1, avec l'effet du tirage sans remise. "
    "Vingt tirages donnent une estimation utile mais encore incertaine de l'ecart-type; l'ecart-type "
    "lui-meme varie sensiblement avec seulement 19 degres de liberte."
)
controle('vingt tirages à chaque taille',
         lambda: len(ecarts_1000) == 20 and len(ecarts_5000) == 20)
controle('les scores tirés sont des pourcentages',
         lambda: 50 < min(ecarts_1000) and max(ecarts_1000) <= 100
                 and 50 < min(ecarts_5000) and max(ecarts_5000) <= 100)
controle('les deux barres ont été mesurées SÉPARÉMENT',
         lambda: abs(ecart_type_1000 - ecart_type_5000) > 1e-9,
         'deux tailles, deux mesures : ne recopiez pas l une dans l autre')
controle('les deux barres sont chiffrées',
         lambda: 0 < ecart_type_1000 < 10 and 0 < ecart_type_5000 < 10 and formule_1000 > 0)
controle('les deux rapports observé / formule sont chiffrés',
         lambda: 0.2 < rapport_1000 < 5 and 0.2 < rapport_5000 < 5,
         'deux nombres : ce que vous avez mesuré, divisé par ce que la formule annonce')
controle('la lecture parle du nombre de tirages',
         lambda: len(lecture) > 60,
         'la barre, le rapport, et la qualité de votre estimation')


  ok        vingt tirages à chaque taille
  ok        les scores tirés sont des pourcentages
  ok        les deux barres ont été mesurées SÉPARÉMENT
  ok        les deux barres sont chiffrées
  ok        les deux rapports observé / formule sont chiffrés
  ok        la lecture parle du nombre de tirages


### Question 6 — La formule qui déborde : ce qu'on ne peut plus tirer  ·  *carnet J1*  ·  *approfondissement*


Reprenez le même tirage, mais avec **25 000 lignes** — presque tout le jeu de contrôle.

À faire :
  1. tirez 20 fois 25 000 lignes, calculez l'écart-type (`ecart_type_25000`) ;
  2. calculez la barre de la formule (`formule_25000`) et le rapport
     (`rapport_25000`) ;
  3. `pourquoi_la_formule_deborde` : en une phrase, ce que la formule suppose et que ce tirage-là
     ne fait plus. Comparez les trois rapports que vous venez de calculer (1 000, 5 000, 25 000) :
     que vaut chacun, et qu'est-ce qui a changé entre le deuxième et le troisième ?
  4. **bonus si vous le voulez** : la **correction de population finie** — l'écart-type d'un tirage
     **sans remise** dans un jeu de `N` lignes se multiplie par racine de (N-n)/(N-1). Calculez-la
     pour n = 25 000, et regardez le rapport qu'elle donne.

À écrire : les 20 scores, l'écart-type, la formule, le rapport, et l'explication.


In [52]:
# Presque toute la validation est tiree sans remise : la formule binomiale seule surestime la barre.
if len(Xval) <= 25000:
    raise ValueError(f"Il faut au moins 25 000 lignes dans Xval; il y en a {len(Xval)}.")
ecarts_25000 = scores_tires(25000, 25000)
ecart_type_25000 = float(np.std(ecarts_25000, ddof=1))
N_validation = len(Xval)
formule_25000 = 100 * np.sqrt(p_validation * (1 - p_validation) / 25000)
rapport_25000 = ecart_type_25000 / formule_25000
correction_population_finie = np.sqrt((N_validation - 25000) / (N_validation - 1))
formule_25000_corrigee = formule_25000 * correction_population_finie
rapport_25000_corrige = (
    ecart_type_25000 / formule_25000_corrigee if formule_25000_corrigee > 0 else float("nan")
)
pourquoi_la_formule_deborde = (
    f"La formule binomiale suppose des observations independantes (comme des tirages avec remise), "
    f"alors que chaque sous-echantillon de 25 000 est tire sans remise dans N={N_validation} lignes : "
    f"il reste peu de nouvelles informations. Les rapports des tailles 1 000, 5 000 et 25 000 sont "
    f"{rapport_1000:.2f}, {rapport_5000:.2f} et {rapport_25000:.2f}; le dernier baisse avec la fraction "
    f"de population observee. La correction finie donne un rapport de {rapport_25000_corrige:.2f}."
)
controle('vingt tirages de 25 000 lignes', lambda: len(ecarts_25000) == 20,
         'la taille demandée est-elle compatible avec celle du jeu de contrôle ?')
controle('la barre observée et la formule ont été chiffrées séparément',
         lambda: 0 < ecart_type_25000 < 10 and 0 < formule_25000 < 10,
         'deux nombres : la mesure observée, puis la formule — pas l un recopié dans l autre')
controle('le rapport est un nombre positif', lambda: 0 < rapport_25000 < 10)
controle('l explication tient en une phrase complète',
         lambda: len(pourquoi_la_formule_deborde) > 60,
         'la formule suppose des tirages indépendants — et là ?')


  ok        vingt tirages de 25 000 lignes
  ok        la barre observée et la formule ont été chiffrées séparément
  ok        le rapport est un nombre positif
  ok        l explication tient en une phrase complète


### Question 7 — Le pas d'apprentissage : qu'est-ce qui change ?  ·  *carnet J1*  ·  *cœur*


Trois valeurs de `learning_rate` — **0,05 / 0,5 / 2** — le reste inchangé. `learning_rate` est le
**pas d'apprentissage** : de combien chaque arbre corrige l'erreur du précédent. C'est le seul
réglage que J1 §6 n'a pas montré (elle n'y a montré que `max_iter`, le nombre d'arbres *demandé*).
Et l'arrêt anticipé s'en mêle : le modèle peut s'arrêter **tout seul** quand sa part interne
cesse de progresser, si bien que le nombre d'arbres *construits* peut différer de celui qu'on
a demandé.

À faire :
  0. **AVANT de mesurer** : `pred_pas` — ce que vous attendez des trois scores et du nombre
     d'arbres construits, avec la raison ;

Puis, dans UN SEUL tableau (`reglages`), une ligne par valeur :
  1. le score de validation, en pourcentage ;
  2. le nombre d'arbres **effectivement construits** (`m.n_iter_`) — pas celui demandé ;
  3. l'écart à la référence, **divisé par la barre de bruit** : combien de barres ?

Puis deux phrases, et elles sont le cœur de la question :
  · `score_separe` : les trois scores sont-ils séparés de **plus de deux barres** ?
  · `ce_qui_borne` : qu'est-ce qui borne **vraiment** ce modèle — le pas, ou autre chose ?
    Appuyez-vous sur ce que vous avez observé, `n_iter_` compris, et dites si cela confirme
    votre `pred_pas`.

À écrire : `pred_pas`, le tableau, `score_separe`, `ce_qui_borne`.


In [53]:
# Prediction formulee avant les mesures : un pas moyen pourrait etre meilleur; le pas tres grand
# devrait degrader le score ou provoquer un arret precoce. Le nombre d'arbres construits peut donc varier.
pred_pas = (
    "Je m'attends a des scores proches pour 0,05 et 0,5, et potentiellement plus faible pour 2, "
    "car un pas trop grand peut depasser une bonne correction. Les pas faibles pourraient demander "
    "plus d'arbres; l'arret anticipe peut toutefois limiter n_iter_ avant max_iter."
)
_lignes_reglages = []
_modeles_reglages = []
for pas in (0.05, 0.5, 2.0):
    m = HistGradientBoostingClassifier(learning_rate=pas, random_state=0).fit(Xtr, ytr)
    score = 100 * m.score(Xval, yval)
    _lignes_reglages.append({
        "learning_rate": pas,
        "score": score,
        "arbres": int(m.n_iter_),
        "en_barres": (score - reference) / erreur_type,
    })
    _modeles_reglages.append(m)
reglages = pd.DataFrame(_lignes_reglages)
display(reglages)
_etendue_en_barres = (reglages["score"].max() - reglages["score"].min()) / erreur_type
score_separe = (
    f"L'etendue des trois scores vaut {_etendue_en_barres:.2f} barre(s) de bruit; "
    + ("elle depasse deux barres, les scores sont donc separes au seuil du cours."
       if _etendue_en_barres > 2 else
       "elle ne depasse pas deux barres : on ne peut pas les distinguer selon la regle du cours.")
)
_resume_arbres = ", ".join(
    f"{r.learning_rate:g}: {int(r.arbres)} arbre(s)" for r in reglages.itertuples()
)
ce_qui_borne = (
    f"Les nombres effectivement construits sont {_resume_arbres}. Le pas ne fixe pas directement "
    "le nombre d'arbres : l'arret anticipe borne l'apprentissage quand la validation interne ne progresse "
    "plus, avec max_iter comme plafond. Si n_iter_ atteint ce plafond, c'est max_iter qui a borne ce modele; "
    "sinon l'arret anticipe l'a fait. Cette observation confirme ou nuance la prediction selon les lignes du tableau."
)
controle('la prédiction est écrite avant le tableau', lambda: len(pred_pas) > 40,
         'ce que vous attendez des scores et des arbres construits, et pourquoi')
controle('trois lignes, une par pas d apprentissage', lambda: len(reglages) == 3)
controle('chaque ligne porte un score en pourcentage',
         lambda: all(50 < r['score'] <= 100 for r in reglages.to_dict('records')),
         'un score de validation, en pourcentage')
controle('chaque ligne porte son NOMBRE D ARBRES CONSTRUITS',
         lambda: all(1 <= r['arbres'] <= 200 for r in reglages.to_dict('records')),
         'm.n_iter_ — le nombre d arbres réellement construits')
controle('chaque ligne rapporte son écart en barres',
         lambda: all(abs(r['en_barres']) < 20 for r in reglages.to_dict('records')),
         'l écart à la référence, divisé par la barre')
controle('les deux phrases sont écrites',
         lambda: len(score_separe) > 40 and len(ce_qui_borne) > 40,
         'deux phrases, appuyées sur votre tableau')


,learning_rate,score,arbres,en_barres
0,0.05,82.541306,73,4.420516
1,0.50,82.165492,14,2.323462
2,2.00,80.764527,10,-5.493959


  ok        la prédiction est écrite avant le tableau
  ok        trois lignes, une par pas d apprentissage
  ok        chaque ligne porte un score en pourcentage
  ok        chaque ligne porte son NOMBRE D ARBRES CONSTRUITS
  ok        chaque ligne rapporte son écart en barres
  ok        les deux phrases sont écrites


### Question 8 — Trois essais de trois natures — et le meilleur de N  ·  *carnet J1*  ·  *cœur*


Au moins **trois essais à vous**, et pas trois fois la même idée. La règle de l'ordre est
**appliquée** : `noter(...)` refuse un essai que vous n'avez pas annoncé.

À faire :
  1. trois essais de **natures différentes** : une **donnée** (une colonne, une correction), un
     **réglage** (un paramètre du modèle), une **famille de modèle** (un autre algorithme) ;
  2. `prevoir(...)` d'abord, puis la mesure dans `essai(...)` — il date la mesure lui-même ;
  3. remplissez `mes_essais` (une **table**, pas une liste) : une ligne par essai, avec **l'essai,
     sa nature, l'attendu, le mesuré, le gain en barres, et le verdict** — « réel » ou « dans le
     bruit ». Le gain se compte **par rapport au modèle de la mise en place** (`score_interne`), pas
     par rapport à la référence : c'est le *changement* qui décide du verdict — et un changement nul
     est une réponse, pas un échec ;
  4. `optimisme` : votre meilleur essai a été choisi **parce qu'il gagne sur cette validation**.
     Écrivez ce que ce choix fait à votre estimation du gain — une phrase, **avant** de mesurer
     (`optimisme`) — puis refaites la mesure du même essai sur un **second découpage** (une autre
     graine) : `verifie_par_second_tirage`, le gain de ce tirage, en barres.

À écrire : `mes_essais`, `optimisme`, `verifie_par_second_tirage`.


In [54]:
# Trois essais de natures distinctes; les attentes sont enregistrees avant de mesurer.
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

noms_essais = {
    "retirer age": "donnee",
    "learning_rate a 0.05": "reglage",
    "arbre de decision": "famille de modele",
}
prevoir("retirer age", "baisse faible ou nulle", "l'age peut porter un signal, mais les arbres tolerent des seuils")
prevoir("learning_rate a 0.05", "score proche ou legerement meilleur", "un pas plus prudent peut stabiliser les corrections")
prevoir("arbre de decision", "score probablement plus faible", "un arbre unique a moins de capacite qu'un ensemble")

def _mesure_sans_age(x_train, x_val, y_train, y_val):
    colonnes = [c for c in x_train.columns if c != "age"]
    return HistGradientBoostingClassifier(random_state=0).fit(
        x_train[colonnes], y_train
    ).score(x_val[colonnes], y_val)

def _mesure_pas_faible(x_train, x_val, y_train, y_val):
    return HistGradientBoostingClassifier(learning_rate=0.05, random_state=0).fit(
        x_train, y_train
    ).score(x_val, y_val)

def _matrice_arbre(x_train, x_val, colonnes):
    train_dummies = pd.get_dummies(x_train, columns=CAT).reindex(columns=colonnes, fill_value=0).fillna(0)
    val_dummies = pd.get_dummies(x_val, columns=CAT).reindex(columns=colonnes, fill_value=0).fillna(0)
    return train_dummies, val_dummies

def _mesure_famille(x_train, x_val, y_train, y_val):
    train_dummies, val_dummies = _matrice_arbre(x_train, x_val, X_num.columns)
    return DecisionTreeClassifier(max_depth=8, min_samples_leaf=100, random_state=0).fit(
        train_dummies, y_train
    ).score(val_dummies, y_val)

score_sans_age = essai("retirer age", lambda: _mesure_sans_age(Xtr, Xval, ytr, yval),
                       "donnee : colonne age retiree", "baisse faible ou nulle car les arbres peuvent ignorer une colonne")
score_pas_faible = essai("learning_rate a 0.05", lambda: _mesure_pas_faible(Xtr, Xval, ytr, yval),
                         "reglage : pas d'apprentissage 0,05", "score proche ou legerement meilleur avec un pas prudent")
score_arbre = essai("arbre de decision", lambda: _mesure_famille(Xtr, Xval, ytr, yval),
                    "famille : arbre de decision", "score probablement plus faible qu'un ensemble d'arbres")

_lignes_essais = [e for e in ESSAIS if e["hypothèse"] in noms_essais]
mes_essais = pd.DataFrame([
    {
        "essai": e["hypothèse"],
        "nature": noms_essais[e["hypothèse"]],
        "attendu": e["attendu"],
        "mesuré": e["mesuré"],
        "gain_en_barres": (e["mesuré"] - score_interne) / erreur_type,
        "verdict": "reel" if abs((e["mesuré"] - score_interne) / erreur_type) >= 2 else "dans le bruit",
    }
    for e in _lignes_essais
])
display(mes_essais)
_meilleur_nom = mes_essais.sort_values("mesuré", ascending=False).iloc[0]["essai"]
optimisme = (
    f"J'ai retenu « {_meilleur_nom} » parce qu'il obtient le meilleur score sur cette validation. "
    "Choisir le gagnant apres plusieurs essais surestime probablement son gain; un second decoupage "
    "independant est necessaire avant de le presenter comme une amelioration."
)

# Second decoupage, graine differente : le modele de base et le meme essai sont reevalues.
Xtr2, Xval2, ytr2, yval2 = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)
score_base2 = 100 * HistGradientBoostingClassifier(random_state=0).fit(Xtr2, ytr2).score(Xval2, yval2)
if _meilleur_nom == "retirer age":
    score_meilleur2 = 100 * _mesure_sans_age(Xtr2, Xval2, ytr2, yval2)
elif _meilleur_nom == "learning_rate a 0.05":
    score_meilleur2 = 100 * _mesure_pas_faible(Xtr2, Xval2, ytr2, yval2)
else:
    score_meilleur2 = 100 * _mesure_famille(Xtr2, Xval2, ytr2, yval2)
barre_second_tirage = 100 * np.sqrt((score_base2 / 100) * (1 - score_base2 / 100) / len(yval2))
verifie_par_second_tirage = (score_meilleur2 - score_base2) / barre_second_tirage
controle('trois essais À VOUS (les amorces ne comptent pas)',
         lambda: sum(1 for e in ESSAIS
                     if e['hypothèse'] not in AMORCES) >= 3,
         'annoncez avec prevoir(...) : noter(...) refuse un essai non annoncé',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('trois natures DIFFÉRENTES',
         lambda: len({str(r['nature']).strip().lower() for r in mes_essais.to_dict('records')}) >= 3,
         'une donnée, un réglage, une famille de modèle')
controle('chaque ligne porte un gain EN BARRES',
         lambda: all(abs(r['gain_en_barres']) < 20 for r in mes_essais.to_dict('records')),
         'le changement divisé par la barre de bruit')
controle('chaque verdict est « réel » ou « dans le bruit »',
         lambda: all(str(r['verdict']).strip().lower() in ('réel', 'reel', 'dans le bruit')
                     for r in mes_essais.to_dict('records')),
         'deux valeurs admises, et rien d autre')
controle('la réponse est écrite avant le second tirage',
         lambda: len(optimisme) > 40,
         'une phrase, écrite AVANT de refaire la mesure')
controle('le second tirage est chiffré en barres',
         lambda: abs(verifie_par_second_tirage) < 20,
         'le même essai, une autre graine : le gain, en barres')


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:45:56
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:45:56
2,retirer age,baisse faible ou nulle,"l'age peut porter un signal, mais les arbres t...",19:46:19


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:45:56
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:45:56
2,retirer age,baisse faible ou nulle,"l'age peut porter un signal, mais les arbres t...",19:46:19
3,learning_rate a 0.05,score proche ou legerement meilleur,un pas plus prudent peut stabiliser les correc...,19:46:19


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:45:56
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:45:56
2,retirer age,baisse faible ou nulle,"l'age peut porter un signal, mais les arbres t...",19:46:19
3,learning_rate a 0.05,score proche ou legerement meilleur,un pas plus prudent peut stabiliser les correc...,19:46:19
4,arbre de decision,score probablement plus faible,un arbre unique a moins de capacite qu'un ense...,19:46:19


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 19:45:56"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.49,"oui, à 19:45:56"
2,retirer age,donnee : colonne age retiree,baisse faible ou nulle car les arbres peuvent ...,82.49,"oui, à 19:46:19"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 19:45:56"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.49,"oui, à 19:45:56"
2,retirer age,donnee : colonne age retiree,baisse faible ou nulle car les arbres peuvent ...,82.49,"oui, à 19:46:19"
3,learning_rate a 0.05,"reglage : pas d'apprentissage 0,05",score proche ou legerement meilleur avec un pa...,82.54,"oui, à 19:46:19"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 19:45:56"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.49,"oui, à 19:45:56"
2,retirer age,donnee : colonne age retiree,baisse faible ou nulle car les arbres peuvent ...,82.49,"oui, à 19:46:19"
3,learning_rate a 0.05,"reglage : pas d'apprentissage 0,05",score proche ou legerement meilleur avec un pa...,82.54,"oui, à 19:46:19"
4,arbre de decision,famille : arbre de decision,score probablement plus faible qu'un ensemble ...,82.46,"oui, à 19:46:19"


,essai,nature,attendu,mesuré,gain_en_barres,verdict
0,retirer age,donnee,baisse faible ou nulle car les arbres peuvent ...,82.49,-0.013301,dans le bruit
1,learning_rate a 0.05,reglage,score proche ou legerement meilleur avec un pa...,82.54,0.265700,dans le bruit
2,arbre de decision,famille de modele,score probablement plus faible qu'un ensemble ...,82.46,-0.180702,dans le bruit


  ok        trois essais À VOUS (les amorces ne comptent pas)
  ok        trois natures DIFFÉRENTES
  ok        chaque ligne porte un gain EN BARRES
  ok        chaque verdict est « réel » ou « dans le bruit »
  ok        la réponse est écrite avant le second tirage
  ok        le second tirage est chiffré en barres


### Question 9 — Ce que le linéaire coûte, en barres — et ce qu'un coefficient ne dit pas  ·  *carnet J2*  ·  *cœur*


**La règle de lecture, posée ici** : un écart inférieur à **deux barres de bruit** n'est pas
une différence. Divisez, et dites combien de barres.

À faire :
  1. l'écart entre les deux familles est mesuré par la mise en place (`score_interne` et
     `score_lineaire`) : `rapport` = cet écart **divisé par la barre**. Est-ce une différence ?
  2. **des coefficients comparables entre eux** : les colonnes n'ont pas les mêmes unités, donc
     la taille brute d'un coefficient ne veut rien dire d'une colonne à l'autre. Deux routes :
     prendre des **indicatrices** (`quartier_…`, `assurance_…` — J2 §1 les construit), ou
     **centrer-réduire** vous-même les colonnes que vous lisez — `(x - x.mean()) / x.std()`, la mise
     à l'échelle que J2 §1 annonce **souhaitable mais non faite**. Nommez les trois colonnes ;
  3. `age` et `age_mois` disent la même chose, `montant_acte` et `cout_acte` aussi : ne les
     utilisez pas pour lire un coefficient, et dites pourquoi (`pourquoi_pas_les_dupliquees`) ;
  4. `quartier_provoque` : le coefficient d'un quartier dit-il que ce quartier **provoque** plus
     d'absences ? **Quelle autre lecture** en est possible ? Une phrase, pas trois ;
  5. `cout_et_rapport` : ce que le linéaire coûte (indicatrices, comblement, mise à
     l'échelle) et ce qu'il rapporte (des coefficients signés, un par colonne).

À écrire : `rapport`, les trois coefficients, `pourquoi_pas_les_dupliquees`, `quartier_provoque`,
`cout_et_rapport`.


In [55]:
# Contraste signé : positif si le linéaire dépasse les arbres, négatif dans le cas inverse.
rapport = (score_lineaire - score_interne) / erreur_type

# Trois indicatrices issues de variables catégorielles : chaque coefficient compare une
# catégorie présente (1) à sa catégorie de référence (0), donc les unités sont comparables.
_coefficients = dict(zip(X_num.columns, modele_lineaire.coef_.ravel()))
_indicatrices = [
    nom for c in CAT for nom in X_num.columns
    if str(nom).startswith(f"{c}_") and nom in _coefficients
]
if len(_indicatrices) < 3:
    raise ValueError("Il faut au moins trois indicatrices catégorielles pour comparer les coefficients.")
trois_coefficients = {nom: float(_coefficients[nom]) for nom in _indicatrices[:3]}

pourquoi_pas_les_dupliquees = (
    "age et age_mois codent la meme quantite dans deux unites, tandis que cout_acte et montant_acte "
    "sont parfaitement redondants : les garder ensemble n'ajoute pas d'information, renforce la "
    "colinearite et rend les coefficients individuels instables ou difficiles a attribuer."
)
quartier_provoque = (
    "Non : le coefficient d'une indicatrice de quartier mesure une association conditionnelle au "
    "quartier de reference, pas un effet causal; la composition des patients, l'offre de soins ou "
    "l'accessibilite peuvent expliquer cette difference observee."
)
cout_et_rapport = (
    "Le modele lineaire demande des indicatrices pour les categories, un traitement coherent des "
    "valeurs manquantes et une mise a l'echelle des variables numeriques; il rapporte des coefficients "
    "signes, interpretes comme des variations de log-odds a autres variables fixes. L'ecart lineaire - "
    f"arbres vaut {rapport:+.2f} barre(s) : "
    + ("il depasse deux barres, donc il est detectable selon la regle du cours."
       if abs(rapport) >= 2 else
       "il reste sous deux barres, donc aucune difference nette n'est etablie.")
)
controle('le rapport écart / barre est un nombre fini',
         lambda: isinstance(rapport, (int, float)) and abs(rapport) < 1e3)
controle('trois coefficients nommés', lambda: len(trois_coefficients) == 3,
         'trois noms de colonnes COMPARABLES entre elles')
controle('aucun coefficient choisi n est une colonne dupliquée',
         lambda: not ({str(c).split('_')[0] for c in trois_coefficients}
                      & {'age', 'montant', 'cout', 'dossier', 'patient'}),
         'age/age_mois et montant_acte/cout_acte disent la même chose')
controle('le refus des dupliquées est argumenté',
         lambda: len(pourquoi_pas_les_dupliquees) > 60)
controle('la lecture du quartier distingue cause et corrélation',
         lambda: len(quartier_provoque) > 60,
         'un coefficient décrit une association, sur des données observées')
controle('le coût et le rapport sont écrits', lambda: len(cout_et_rapport) > 60)


  ok        le rapport écart / barre est un nombre fini
  ok        trois coefficients nommés
  ok        aucun coefficient choisi n est une colonne dupliquée
  ok        le refus des dupliquées est argumenté
  ok        la lecture du quartier distingue cause et corrélation
  ok        le coût et le rapport sont écrits


### Question 10 — Votre feature — différente de celle du carnet, et son contournement  ·  *carnet J2*  ·  *approfondissement*


**Écrivez une feature que le carnet n'a pas écrite.** `delai_long` est l'exemple de la séance 2,
et la recopier n'apprend rien : votre feature doit découler d'**un de vos constats de la
question 3**, et vous devez dire lequel.

À faire :
  1. dans `ajouter_features` — **la seule cellule de la mise en place faite pour être modifiée** —
     écrivez votre feature, et notez `constat_d_origine` (l'observation du constat qui l'a
     motivée, telle qu'elle est écrite dans le tableau) ;
  2. `prevoir(...)`, puis `remesurer()` : la chaîne passe par `preparer()`, donc `X` reçoit votre
     feature **une seule fois**, et `modele` est mis à jour ;
  3. `gain_de_ma_feature`, en barres de bruit — **un gain de zéro est une réponse** : la table est
     sur un plateau, et une feature raisonnable peut n'y rien changer ;
  4. **contournez `preparer()`** : appelez `modele.predict` sur une table préparée à la main en
     oubliant une étape → `score_contourne`, `ecart_du_contournement` (en barres), et
     `quelle_etape_oubliee`.

À écrire : la feature, son constat d'origine, le gain, le contournement et ce qu'il a produit.


In [56]:
constat_d_origine = "des ages impossibles sont presents"

# Cette annonce precede la mesure. On compare le score de la chaine avec feature au score de base.
prevoir(
    "indicateur_age_sentinelle",
    "gain faible ou nul, inferieur a deux barres",
    "les arbres peuvent deja separer les ages codes -1 par un seuil",
)
score_ma_feature = essai(
    "indicateur_age_sentinelle",
    lambda: remesurer() / 100,
    "ajout de age_sentinelle via preparer()",
    "le signal existe deja dans age; le nouvel indicateur peut ne rien ajouter",
)
gain_de_ma_feature = (score_ma_feature - score_interne) / erreur_type

# Contournement volontaire : preparation manuelle, mais etape de calcul de la feature omise.
# On met un zero de remplacement pour garder la forme attendue et mesurer l'effet de l'oubli.
X_contourne = X_sans_features.loc[Xval.index].copy()
for c in CAT:
    if c in X_contourne.columns:
        X_contourne[c] = X_contourne[c].cat.set_categories(X[c].cat.categories)
X_contourne["age_sentinelle"] = np.int8(0)
X_contourne = X_contourne.reindex(columns=X.columns)
pred_contourne = modele.predict(X_contourne)
score_contourne = 100 * (pred_contourne == yval).mean()
ecart_du_contournement = (score_contourne - score_ma_feature) / erreur_type
quelle_etape_oubliee = (
    "Le calcul de age_sentinelle dans ajouter_features (appele par preparer) a ete omis; "
    "la colonne de remplacement mise a zero garde la forme, mais efface le signal -1."
)
controle('la feature s inspire d un constat de la question 3',
         lambda: constat_d_origine in [c['observation'] for c in CONSTATS],
         'recopiez l observation exacte du constat qui motive votre feature')
controle('votre feature a enrichi la table', lambda: X.shape[1] > n_colonnes_avant,
         'elle entre par la chaîne, pas à la main',
         des_que=lambda: not ajouter_features(X_sans_features).equals(X_sans_features))
controle('le gain est chiffré en barres', lambda: abs(gain_de_ma_feature) < 20)
controle('l écart du contournement est chiffré en barres',
         lambda: abs(ecart_du_contournement) < 20,
         'un modèle entraîné sur une table et interrogé sur une autre')
controle('l étape oubliée est nommée', lambda: len(quelle_etape_oubliee) > 20,
         'nommez l étape manquante : c est elle, le bug silencieux')


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,19:45:56
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",19:45:56
2,retirer age,baisse faible ou nulle,"l'age peut porter un signal, mais les arbres t...",19:46:19
3,learning_rate a 0.05,score proche ou legerement meilleur,un pas plus prudent peut stabiliser les correc...,19:46:19
4,arbre de decision,score probablement plus faible,un arbre unique a moins de capacite qu'un ense...,19:46:19
5,indicateur_age_sentinelle,"gain faible ou nul, inferieur a deux barres",les arbres peuvent deja separer les ages codes...,19:46:30


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 19:45:56"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.49,"oui, à 19:45:56"
2,retirer age,donnee : colonne age retiree,baisse faible ou nulle car les arbres peuvent ...,82.49,"oui, à 19:46:19"
3,learning_rate a 0.05,"reglage : pas d'apprentissage 0,05",score proche ou legerement meilleur avec un pa...,82.54,"oui, à 19:46:19"
4,arbre de decision,famille : arbre de decision,score probablement plus faible qu'un ensemble ...,82.46,"oui, à 19:46:19"
5,indicateur_age_sentinelle,ajout de age_sentinelle via preparer(),le signal existe deja dans age; le nouvel indi...,82.49,"oui, à 19:46:30"


  ok        la feature s inspire d un constat de la question 3
  ok        votre feature a enrichi la table
  ok        le gain est chiffré en barres
  ok        l écart du contournement est chiffré en barres
  ok        l étape oubliée est nommée


### Question 11 — La colonne qui disparaît : la trouver, mesurer son gain, et le voir s'évaporer  ·  *carnet J2*  ·  *cœur*


**Cherchez-la.** J1 l'a rangée dans `A_ECARTER` — parmi dix autres, et sans dire pourquoi.
Ce qui la trahit est un **écart de remplissage** entre `train` et `test`.

Le gain et la simulation se mesurent sur **`mon_train`** — votre échantillon, comme à la
question 2 : un gain recopié d'un voisin ne vaudra pas le vôtre.

À faire :
  1. calculez, **colonne par colonne**, l'écart de remplissage entre `train` et `test`, triez, et
     nommez celle qui se détache (`colonne_trouvee`) avec son écart (`ecart_remplissage`, en
     points). La **deuxième** du tri mérite la même attention : dites pour elle aussi pourquoi elle
     n'est pas la réponse (`pourquoi_pas_la_derivee`) ;
  2. `remplissage_du_test` : quelle **part du test** porte encore une valeur dans cette colonne ?
  3. `gain_interne` : le même modèle, avec et sans cette colonne, sur la **même** validation —
     le gain en points, et en barres (`gain_en_barres`) ;
  4. **la simulation, et c'est le cœur** : entraînez avec la colonne, puis rendez-la manquante
     dans la validation **pour une part de lignes égale à celle du test** (le taux de trous du
     test). `score_simule`, `chute_en_barres` : que reste-t-il du gain ?
  5. `decision_A_ECARTER` : l'écarter était-il la bonne décision ? Était-ce pour la bonne
     raison ? Une phrase pour chacune des deux questions.

À écrire : la colonne, l'écart, la part du test, le gain interne, la simulation, la décision.


In [57]:
# Ecart absolu de taux de remplissage, sur toutes les colonnes communes (hors cible).
colonnes_communes = [c for c in train.columns if c in test.columns and c != CIBLE]
ecarts = pd.Series({
    c: 100 * abs(train[c].notna().mean() - test[c].notna().mean())
    for c in colonnes_communes
}, name="ecart_remplissage").sort_values(ascending=False)
if len(ecarts) < 2:
    raise ValueError("Il faut au moins deux colonnes communes pour comparer le remplissage.")
colonne_trouvee = ecarts.index[0]
ecart_remplissage = float(ecarts.iloc[0])
colonne_derivee = ecarts.index[1]
remplissage_derivee_train = 100 * train[colonne_derivee].notna().mean()
remplissage_derivee_test = 100 * test[colonne_derivee].notna().mean()
pourquoi_pas_la_derivee = (
    f"La deuxieme colonne du tri est {colonne_derivee!r} (ecart {ecarts.iloc[1]:.2f} points; "
    f"remplie a {remplissage_derivee_train:.2f} % dans train et {remplissage_derivee_test:.2f} % dans test). "
    "Son taux ne suffit pas a etablir qu'elle disparait completement; il faut distinguer une derive "
    "de remplissage d'une colonne entierement absente et examiner ce que sa valeur signifie."
)
remplissage_du_test = 100 * test[colonne_trouvee].notna().mean()
if colonne_trouvee not in mon_train.columns:
    raise ValueError(f"La colonne la plus divergente {colonne_trouvee!r} n'est pas dans mon_train.")

# Comparaison sur le meme split de mon_train, avec et sans la colonne mise de cote par J1.
X_q11_base, _, _ = matrice(mon_train)
y_q11 = mon_train[CIBLE]
if colonne_trouvee in X_q11_base.columns:
    X_q11_base = X_q11_base.drop(columns=[colonne_trouvee])
valeurs_q11 = mon_train[colonne_trouvee]
if not pd.api.types.is_numeric_dtype(valeurs_q11):
    valeurs_q11 = valeurs_q11.astype("category")
X_q11_avec = X_q11_base.copy()
X_q11_avec[colonne_trouvee] = valeurs_q11
X11_tr, X11_val, y11_tr, y11_val = train_test_split(
    X_q11_avec, y_q11, test_size=0.2, stratify=y_q11, random_state=0
)
X11_tr_sans = X_q11_base.loc[X11_tr.index]
X11_val_sans = X_q11_base.loc[X11_val.index]
modele11_avec = HistGradientBoostingClassifier(random_state=0).fit(X11_tr, y11_tr)
modele11_sans = HistGradientBoostingClassifier(random_state=0).fit(X11_tr_sans, y11_tr)
score11_avec = 100 * modele11_avec.score(X11_val, y11_val)
score11_sans = 100 * modele11_sans.score(X11_val_sans, y11_val)
gain_interne = score11_avec - score11_sans
barre11 = 100 * np.sqrt((score11_avec / 100) * (1 - score11_avec / 100) / len(y11_val))
gain_en_barres = gain_interne / barre11

# Simulation de la proportion de valeurs manquantes observee dans le vrai test.
taux_trous_test = 1 - remplissage_du_test / 100
X11_simule = X11_val.copy()
n_a_masquer = int(round(taux_trous_test * len(X11_simule)))
indices_a_masquer = X11_simule.index[:0]
if n_a_masquer:
    indices_a_masquer = X11_simule.sample(n=n_a_masquer, random_state=0).index
serie_simulee = X11_simule[colonne_trouvee]
if isinstance(serie_simulee.dtype, pd.CategoricalDtype):
    valeurs_objet = serie_simulee.astype(object)
    valeurs_objet.loc[indices_a_masquer] = np.nan
    X11_simule[colonne_trouvee] = pd.Categorical(
        valeurs_objet, categories=serie_simulee.cat.categories
    )
else:
    X11_simule.loc[indices_a_masquer, colonne_trouvee] = np.nan
score_simule = 100 * modele11_avec.score(X11_simule, y11_val)
chute_en_barres = (score_simule - score11_avec) / barre11

decision_A_ECARTER = (
    f"Oui, l'ecarter est prudent : le test ne conserve que {remplissage_du_test:.2f} % de ses valeurs, "
    f"et le gain interne est de {gain_interne:+.2f} points ({gain_en_barres:+.2f} barre(s)); "
    f"quand on simule ce taux de trous, le score devient {score_simule:.2f} % "
    f"({chute_en_barres:+.2f} barre(s) par rapport au modele avec colonne). "
    + ("La raison est la disponibilite en production, pas l'absence de signal en train."
       if "absence" in str(colonne_trouvee).lower() or "motif" in str(colonne_trouvee).lower()
       else "La raison doit etre confirmee par le sens metier de la colonne, pas par son seul taux de remplissage.")
)
display(ecarts.head(10).to_frame())
controle('l écart de remplissage est un pourcentage', lambda: 0 <= ecart_remplissage <= 100)
controle('la colonne nommée est bien celle de VOTRE tri',
         lambda: colonne_trouvee == ecarts.index[0],
         'la première de votre propre tri : l écart est net, il n y a pas d hésitation')
controle('la colonne qui dérive est écartée, avec sa raison',
         lambda: len(pourquoi_pas_la_derivee) > 40,
         'une dérive n est pas une disparition : chiffrez-la')
controle('la part du test encore renseignée est un pourcentage',
         lambda: 0 <= remplissage_du_test <= 100,
         'ce qui survit dans le test : un pourcentage, la part encore renseignée')
controle('le gain interne et sa simulation sont chiffrés',
         lambda: abs(gain_en_barres) < 40 and abs(chute_en_barres) < 60,
         'deux nombres, chacun divisé par la barre')
controle('la simulation a été mesurée sur un score à part',
         lambda: 0 < abs(chute_en_barres) < 60,
         'le même modèle, la validation rendue trouée : comparez les deux scores')
controle('la décision est argumentée, sur les deux questions',
         lambda: len(decision_A_ECARTER) > 80,
         'la bonne décision, et la bonne raison')


,ecart_remplissage
canal_prise,99.502519
motif_absence,0.752257
rappel_envoye,0.332756
constantes_relevees,0.269844
solde_credit,0.022453
date_naissance,0.003837
age,0.000000
district,0.000000
clinique,0.000000
patient_id,0.000000


  ok        l écart de remplissage est un pourcentage
  ok        la colonne nommée est bien celle de VOTRE tri
  ok        la colonne qui dérive est écartée, avec sa raison
  ok        la part du test encore renseignée est un pourcentage
  ok        le gain interne et sa simulation sont chiffrés
  ok        la simulation a été mesurée sur un score à part
  ok        la décision est argumentée, sur les deux questions


### Question 12 — Votre outil : `suspects(df, cible)`  ·  *carnet J2*  ·  *cœur*


Le carnet a montré **deux signes** d'une colonne qui connaît la réponse, mais il l'a fait une
colonne à la fois. Écrivez l'**outil** qui les cherche toutes — vous le réutiliserez en projet.

À faire :
  1. `suspects(df, cible)` rend une **table**, une ligne par colonne suspecte, avec au moins :
     le nom, son **écart de remplissage** entre les deux groupes de la cible, et le **signe**
     qui l'a trahie ;
     — **signe 1, le VIDE** : une colonne remplie d'un côté et vide de l'autre (écart de
       remplissage de plus de 50 points) ;
     — **signe 2, la VALEUR** : une colonne remplie **des deux côtés** dont une valeur ne vit
       que dans **un** groupe ;
     — votre outil doit **ignorer l'identifiant** et la cible elle-même ;
  2. appliquez-le à `train` : `table_suspects` ;
  3. Rendez `classement_suspects` — une table d'**une ligne par colonne signalée**, avec deux
     colonnes : `verdict` (à choisir dans **fuite** / **pas une fuite** / **à discuter**) et
     `justification` — et justifiez par **ce que la valeur veut dire**, jamais par le fait qu'elle
     est constante ;
  4. **le troisième signe, celui que les deux premiers ne voient pas** : une valeur
     **impossible** (une **sentinelle** : un code mis à la place d'une mesure manquante — le −1
     d'`age` en séance 1 en est une) portée par **peu** de lignes. `sentinelle_trouvee`
     (la colonne et la valeur), `lignes_porteuses` (combien de lignes), `taux_sur_ces_lignes`
     (le taux d'absence sur ces lignes, comparé au taux global), et `verdict_sentinelle` : fuite,
     ou défaut de saisie ?

À écrire : la fonction, `table_suspects`, `classement_suspects`, la sentinelle et son verdict.


In [58]:
# Outil de repérage des colonnes dont remplissage ou valeurs dépendent du groupe cible.
def suspects(df, cible):
    """Une ligne par colonne suspecte, avec l'indice qui la signale."""
    groupes = list(pd.unique(df[cible].dropna()))
    if len(groupes) != 2:
        raise ValueError(f"Il faut exactement deux groupes non vides dans {cible!r}.")
    masques = [df[cible].eq(groupe) for groupe in groupes]
    lignes = []
    for colonne in df.columns:
        if colonne in (ID, cible):
            continue
        serie = df[colonne]
        taux = [100 * serie.loc[masque].notna().mean() for masque in masques]
        ecart = abs(taux[0] - taux[1])
        signes = []
        if ecart > 50:
            signes.append(
                f"VIDE : rempli a {taux[0]:.1f} % dans {groupes[0]!r} et {taux[1]:.1f} % dans {groupes[1]!r}"
            )
        if taux[0] > 0 and taux[1] > 0:
            valeurs_exclusives = []
            for valeur in pd.unique(serie.dropna()):
                presents = [bool((serie.loc[masque] == valeur).any()) for masque in masques]
                if sum(presents) == 1:
                    valeurs_exclusives.append(valeur)
                    if len(valeurs_exclusives) == 3:
                        break
            if valeurs_exclusives:
                signes.append(f"VALEUR : {valeurs_exclusives!r} ne vit que dans un groupe")
        if signes:
            lignes.append({
                "colonne": colonne,
                "ecart_remplissage": float(ecart),
                "signe": "; ".join(signes),
            })
    return pd.DataFrame(lignes, columns=["colonne", "ecart_remplissage", "signe"])

table_suspects = suspects(train, CIBLE)

def _classer_colonne_suspecte(nom):
    nom_minuscule = str(nom).lower()
    exemple = train[nom].dropna().iloc[0] if train[nom].notna().any() else "aucune valeur renseignee"
    if "motif_absence" in nom_minuscule or "date_absence" in nom_minuscule:
        return "fuite", (
            f"La valeur {exemple!r} de {nom} decrit le motif ou la date d'une absence deja constatee; "
            "cette information n'est disponible qu'apres le rendez-vous et revele l'issue."
        )
    if "id" in nom_minuscule or "identifiant" in nom_minuscule:
        return "à discuter", (
            f"La valeur {exemple!r} de {nom} identifie un dossier ou une personne, pas un comportement "
            "mesure au moment de la prediction; elle peut memoriser des groupes et doit etre ecartee "
            "ou auditee comme proxy avant toute mise en production."
        )
    return "pas une fuite", (
        f"La valeur {exemple!r} de {nom} represente une information de la ligne; son association avec "
        "la cible ne prouve pas qu'elle est observee apres l'issue. Il faut verifier sa disponibilite "
        "au moment de la prediction avant de l'utiliser."
    )

classement_suspects = pd.DataFrame(
    [_classer_colonne_suspecte(nom) for nom in table_suspects["colonne"]],
    columns=["verdict", "justification"],
    index=table_suspects["colonne"],
)

# L'age code -1 est la sentinelle impossible reperee dans le cours : c'est une saisie manquante codee.
sentinelle_trouvee = ("age", -1)
masque_sentinelle = pd.to_numeric(train["age"], errors="coerce").eq(-1)
lignes_porteuses = int(masque_sentinelle.sum())
if not lignes_porteuses:
    raise ValueError("La sentinelle attendue age = -1 n'a pas été trouvée dans train.")
taux_sur_ces_lignes = 100 * train.loc[masque_sentinelle, CIBLE].mean()
taux_global_absence = 100 * train[CIBLE].mean()
verdict_sentinelle = (
    f"Defaut de saisie, pas fuite : -1 code une mesure d'age manquante/impossible sur {lignes_porteuses} lignes; "
    f"leur taux d'absence est {taux_sur_ces_lignes:.2f} % contre {taux_global_absence:.2f} % globalement. "
    "Cette association peut signaler un processus de saisie particulier, mais la sentinelle ne contient "
    "pas en elle-meme une information observee apres le rendez-vous."
)
display(table_suspects)
display(classement_suspects)
controle('`suspects` rend une table, pas une liste de mots',
         lambda: hasattr(table_suspects, 'columns'),
         'une ligne par colonne suspecte : un nom, un écart de remplissage, un signe')
controle('l identifiant et la cible sont exclus',
         lambda: not ({ID, CIBLE} & set(table_suspects.iloc[:, 0])),
         'un identifiant sépare tout : il n apprend rien')
controle('le signe du VIDE apparaît au moins une fois',
         lambda: any('vide' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie d un côté, vide de l autre')
controle('le signe de la VALEUR apparaît au moins une fois',
         lambda: any('valeur' in str(v).lower()
                     for v in table_suspects.to_numpy().ravel()),
         'une colonne remplie partout, dont une valeur ne vit que dans un groupe')
controle('chaque verdict a été choisi, et le tri n est pas uniforme',
         lambda: len(set(str(v).strip().lower() for v in classement_suspects['verdict'])) >= 2,
         'la même réponse recopiée sur toutes les lignes n est pas un tri')
controle('chaque colonne signalée est classée, et chaque verdict argumenté',
         lambda: len(classement_suspects) == len(table_suspects)
                 and all(len(str(x)) > 30 for x in classement_suspects['justification']),
         'ce que la valeur VEUT DIRE, pas le fait qu elle est constante')
controle('la sentinelle est nommée, chiffrée, et jugée',
         lambda: len(str(sentinelle_trouvee)) > 5 and 0 < lignes_porteuses < 5000
                 and 0 <= taux_sur_ces_lignes <= 100 and len(verdict_sentinelle) > 40,
         'une valeur impossible, sur peu de lignes : fuite, ou saisie ?')


,colonne,ecart_remplissage,signe
0,dossier_id,0.000000,"VALEUR : ['DOS-0127912', 'DOS-0002883', 'DOS-0..."
1,patient_id,0.000000,"VALEUR : ['P109083', 'P036240', 'P033484'] ne ..."
2,duree_consultation,0.000000,"VALEUR : [np.int64(29), np.int64(20), np.int64..."
3,motif_absence,100.000000,VIDE : rempli a 100.0 % dans np.int64(1) et 0....
4,taille_cm,0.000000,"VALEUR : ['141,1', '197,7', '196,7'] ne vit qu..."
5,date_prise,0.000000,"VALEUR : ['2025-11-11T13:00', '2024-12-15T06:0..."
6,encours_1_mois,0.000000,"VALEUR : [np.float64(229.83), np.float64(215.2..."
7,encours_3_mois,0.000000,"VALEUR : [np.float64(227.46), np.float64(213.0..."
8,encours_6_mois,0.000000,"VALEUR : [np.float64(214.54), np.float64(89.92..."
9,solde_credit,0.094153,"VALEUR : [np.float64(2562.34), np.float64(1662..."


,verdict,justification
colonne,,
dossier_id,à discuter,La valeur 'DOS-0127912' de dossier_id identifi...
patient_id,à discuter,La valeur 'P070425' de patient_id identifie un...
duree_consultation,pas une fuite,La valeur np.int64(-1) de duree_consultation r...
motif_absence,fuite,La valeur 'non précisé' de motif_absence decri...
taille_cm,pas une fuite,"La valeur '161,9' de taille_cm represente une ..."
date_prise,pas une fuite,La valeur '2023-07-02T11:00' de date_prise rep...
encours_1_mois,pas une fuite,La valeur np.float64(64.4) de encours_1_mois r...
encours_3_mois,pas une fuite,La valeur np.float64(60.6) de encours_3_mois r...
encours_6_mois,pas une fuite,La valeur np.float64(73.0) de encours_6_mois r...


  ok        `suspects` rend une table, pas une liste de mots
  ok        l identifiant et la cible sont exclus
  ok        le signe du VIDE apparaît au moins une fois
  ok        le signe de la VALEUR apparaît au moins une fois
  ok        chaque verdict a été choisi, et le tri n est pas uniforme
  ok        chaque colonne signalée est classée, et chaque verdict argumenté
  ok        la sentinelle est nommée, chiffrée, et jugée


### Question 13 — Le rappel : cause, ou autre chose ? — et l'expérience qui trancherait  ·  *carnet J2*  ·  *approfondissement*


Un rappel est envoyé, et l'absence est **plus** fréquente avec le rappel. Faut-il en conclure
que le rappel fait manquer les rendez-vous ?

**Tout se mesure sur `mon_train`** — votre échantillon, comme aux questions 2 et 11.

À faire :
  1. **AVANT de regarder les tranches** : `annonce` — le rappel aide-t-il, ou nuit-il, une fois la
     comparaison faite **à délai égal** ? Écrivez votre prédiction, et ce qui vous ferait changer
     d'avis ;
  2. le taux d'absence avec et sans rappel, **globalement** : `ecart_global` (en points) ;
  3. **à délai égal** : par tranches de délai, le taux avec et sans rappel dans **chaque** tranche,
     et pour chaque écart **sa barre de bruit** (`tranches`) — sans la barre, la conclusion
     contredit la règle du cours : un écart de moins de deux barres n'est pas un écart. Comptez
     les **effectifs** : toutes les tranches ne portent pas les deux groupes, la plus courte n'en
     porte qu'un — et sur **votre** échantillon, certaines tomberont **sous les 500 lignes** :
     celles-là ne se comparent pas, mettez-les dans `tranches_exclues` en disant pourquoi ;
  4. `variable_qui_explique` : la variable — s'il y en a une — qui explique l'écart global, avec
     le chiffre qui le montre (`delai_median_avec`, `delai_median_sans`) ; s'il n'y en a pas,
     dites-le ;
  5. `experience_qui_trancherait` : quelle expérience **trancherait vraiment** ? Décrivez-la en
     trois lignes — ce qu'on tirerait au hasard, ce qu'on comparerait, et pourquoi la comparaison
     serait alors honnête.

À écrire : l'annonce, l'écart global, les tranches avec leur barre, la variable, l'expérience.


In [59]:
# TODO : votre annonce, AVANT de regarder les tranches.
# annonce = … (à vous)
# TODO : l'écart global, puis les tranches AVEC leur barre de bruit.
# ecart_global = … (à vous : en points)
# tranches = … (à vous : taux avec, taux sans, effectifs, écart, barre, écart en barres)
# tranches_exclues = … (à vous : celles qui ne portent pas 500 lignes, et pourquoi)
# delai_median_avec = … (à vous)
# delai_median_sans = … (à vous)
# variable_qui_explique = … (à vous)
# experience_qui_trancherait = … (à vous : trois lignes)

controle('l annonce est écrite avant les tranches', lambda: len(annonce) > 60,
         'ce que vous attendez, ET ce qui vous ferait changer d avis')
controle('l écart global est chiffré, en points', lambda: abs(ecart_global) < 100,
         'le taux avec rappel moins le taux sans rappel, en points')
controle('les tranches portent leur barre, pas seulement leur écart',
         lambda: len(tranches) >= 2 and all('barre' in r for r in tranches),
         'sans la barre, un écart de 2 points ne veut rien dire')
controle('les tranches trop fines de VOTRE échantillon sont mises à part',
         lambda: hasattr(tranches_exclues, '__len__')
                 and all(r.get('lignes', 0) >= 500 for r in tranches),
         'sous 500 lignes, on ne conclut rien : nommez-les au lieu de les comparer')
controle('chaque tranche comparée porte son écart et son effectif',
         lambda: all(abs(r['ecart']) > 0 and r.get('lignes', 0) > 0 for r in tranches),
         'un écart chiffré par tranche — et le nombre de lignes qui le porte')
controle('les effectifs sont comptés',
         lambda: all(r.get('lignes', 0) > 0 for r in tranches),
         'une tranche sans les deux groupes ne se compare pas')
controle('la variable qui explique est nommée, et les deux médianes sont chiffrées',
         lambda: len(str(variable_qui_explique)) > 3 and delai_median_avec >= 0
                 and delai_median_sans >= 0,
         'les deux médianes du délai, en jours, avec et sans rappel')
controle('l expérience qui trancherait est décrite en trois lignes',
         lambda: len(experience_qui_trancherait) > 120,
         'ce qu on tire au hasard, ce qu on compare, et pourquoi c est honnête')


  (pas encore écrit)  l annonce est écrite avant les tranches
  (pas encore écrit)  l écart global est chiffré, en points
  (pas encore écrit)  les tranches portent leur barre, pas seulement leur écart
  (pas encore écrit)  les tranches trop fines de VOTRE échantillon sont mises à part
  (pas encore écrit)  chaque tranche comparée porte son écart et son effectif
  (pas encore écrit)  les effectifs sont comptés
  (pas encore écrit)  la variable qui explique est nommée, et les deux médianes sont chiffrées
  (pas encore écrit)  l expérience qui trancherait est décrite en trois lignes


### Question 14 — Le rapport final — et une hypothèse testée sous son propre nom  ·  *carnet J1 + J2*  ·  *cœur*


Les trois tableaux se remplissent **dans les questions** ; cette question-ci produit les
hypothèses, et **en teste une**.

À faire :
  1. écrivez **trois** hypothèses de features avec `hypothese(...)` : ce qu'elle raconte (avec le
     **signe** attendu), comment on la teste, et ce qui la **contredirait** — un résultat précis,
     pas « si ça ne marche pas » ;
  2. **testez-en une** avec `essai(...)`, sous **le même nom** que l'hypothèse : la mesure, puis
     `confirmee_ou_contredite` — « confirmée » ou « contredite », jugée sur le critère que
     l'hypothèse avait annoncé **avant** la mesure ;
  3. un « contredite » n'est pas un échec : c'est le seul des deux verdicts qui apprend quelque
     chose. Écrivez ce que vous en faites (`suite_donnee`).

À écrire : les trois hypothèses, le test, le verdict, la suite donnée.


In [60]:
# TODO : trois hypothèses, ici.
# hypothese("ma feature", "ce qu'elle raconte, avec le signe", "comment je la teste",
#           "ce qui la contredirait")
# TODO : une hypothèse TESTÉE, sous son propre nom.
# essai("ma feature", lambda: modele.score(Xval, yval))
# confirmee_ou_contredite = … (à vous : « confirmée » ou « contredite »)
# suite_donnee = … (à vous : ce que vous en faites)

controle('trois hypothèses', lambda: len(HYPOTHESES) == 3,
         des_que=lambda: len(HYPOTHESES) > 0)
controle('chacune dit ce qui la contredirait, précisément',
         lambda: all(len(str(h['contredite si'])) > 40 for h in HYPOTHESES),
         'un résultat précis, pas « si ça ne marche pas »',
         des_que=lambda: len(HYPOTHESES) > 0)
controle('une hypothèse a été testée sous son propre nom',
         lambda: any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'le tableau d essais doit porter le nom d une de vos hypothèses',
         des_que=lambda: any(e['hypothèse'] not in AMORCES for e in ESSAIS))
controle('le verdict est confirmée ou contredite',
         lambda: str(confirmee_ou_contredite).strip().lower().rstrip('e')
                 in ('confirmé', 'confirm', 'contredit'),
         'deux verdicts admis, jugés sur le critère annoncé')
controle('la suite donnée est écrite', lambda: len(suite_donnee) > 60)
controle('les trois tableaux sont remplis avant le rapport',
         lambda: len(CONSTATS) == 5 and len(HYPOTHESES) == 3 and len(ESSAIS) >= 3,
         des_que=lambda: len(HYPOTHESES) > 0)


  (pas encore écrit)  trois hypothèses
  (pas encore écrit)  chacune dit ce qui la contredirait, précisément
  REVOIR    une hypothèse a été testée sous son propre nom   ->   le tableau d essais doit porter le nom d une de vos hypothèses
  (pas encore écrit)  le verdict est confirmée ou contredite
  (pas encore écrit)  la suite donnée est écrite
  (pas encore écrit)  les trois tableaux sont remplis avant le rapport


### Question 15 — La méthode hors de ce jeu de données : une table de crédit  ·  *carnet J2*  ·  *cœur*


La question que le cours doit rendre possible : **est-ce que la méthode survit ailleurs ?**

Voici l'en-tête d'une autre table — des demandes de crédit :

| colonne | ce qu'elle contient |
|---|---|
| `id_client` | l'identifiant du client |
| `date_defaut` | la date du défaut de paiement — renseignée **seulement** si l'emprunteur a fait défaut |
| `statut_6_mois` | « à jour / 30 jours / 90 jours », relevé **six mois après** l'octroi |
| `montant` | le montant demandé, en euros |
| `canal` | le canal de la demande : agence, web, téléphone |
| `score_interne` | le score de risque calculé **au moment de la demande** |

À faire : classez **chaque** colonne dans **une** des quatre familles de la séance 2, et
justifiez chacune en une phrase :
  · **identifiant** — elle désigne une ligne, elle ne la décrit pas ;
  · **connue après l'issue** — sa valeur n'existe pas encore au moment de décider ;
  · **risque de disparition** — elle existe ici et pourrait ne plus être alimentée en production ;
  · **utilisable** — disponible au moment de la décision, et elle décrit la ligne.

À écrire : `familles` (une famille par colonne) et `justifications` (une phrase par colonne).


In [61]:
# TODO : une famille par colonne, du vocabulaire fermé ci-dessus.
# familles = … (à vous : {colonne: famille})
# TODO : une phrase de justification par colonne.
# justifications = … (à vous : {colonne: phrase})

COLONNES_CREDIT = ['id_client', 'date_defaut', 'statut_6_mois', 'montant', 'canal',
                   'score_interne']
FAMILLES_ATTENDUES = ['identifiant', "connue après l'issue", 'risque de disparition',
                      'utilisable']

controle('les six colonnes sont classées',
         lambda: set(familles) == set(COLONNES_CREDIT),
         'les six colonnes de l en-tête, ni plus ni moins')
controle('le vocabulaire est celui des quatre familles',
         lambda: set(familles.values()) <= set(FAMILLES_ATTENDUES),
         'quatre familles admises, mot pour mot')
controle('la colonne du défaut n est pas classée « utilisable »',
         lambda: familles['date_defaut'] != 'utilisable',
         'elle n existe qu après coup : quelle famille lui revient ?')
controle('l identifiant est reconnu comme tel',
         lambda: familles['id_client'] == 'identifiant')
controle('le score calculé à la demande reste utilisable',
         lambda: familles['score_interne'] == 'utilisable',
         'disponible au moment de décider : c est la définition')
controle('les six justifications sont écrites, une phrase chacune',
         lambda: set(justifications) == set(COLONNES_CREDIT)
                 and all(len(str(x)) > 30 for x in justifications.values()))


  (pas encore écrit)  les six colonnes sont classées
  (pas encore écrit)  le vocabulaire est celui des quatre familles
  (pas encore écrit)  la colonne du défaut n est pas classée « utilisable »
  (pas encore écrit)  l identifiant est reconnu comme tel
  (pas encore écrit)  le score calculé à la demande reste utilisable
  (pas encore écrit)  les six justifications sont écrites, une phrase chacune


## Les trois tableaux à rendre

C'est la pièce principale du devoir. Les fonctions `constat(...)`, `hypothese(...)` et
`noter(...)` sont définies dans la mise en place : appelez-les, elles affichent le tableau.

### 1. Cinq constats chiffrés

**Cinq** lignes, dont **deux au moins mesurées par vous** (les trois de la séance sont déjà
écrites). Chaque ligne a quatre morceaux : l'observation, le chiffre, **par rapport à quoi**, et
ce que vous en faites. Un constat sans comparaison ne compte pas.

### 2. Trois hypothèses de features

Une hypothèse est une **prédiction qu'on peut tester** : ce qu'elle raconte (avec le signe),
comment on la teste, et **ce qui la contredirait** — un résultat précis. La question 14 en teste
une sous son propre nom.

### 3. Le tableau d'essais

Une ligne par changement essayé. L'**attendu** porte un signe **et** une raison, et le **gain en
barres** décide du verdict : « réel » ou « dans le bruit ».

La cellule qui suit affiche les trois tableaux tels que vous les rendez.


In [62]:
print("CONSTATS CHIFFRÉS \n")
display(pd.DataFrame(CONSTATS))
print("HYPOTHÈSES DE FEATURES \n")
display(pd.DataFrame(HYPOTHESES))
display(pd.DataFrame(ESSAIS))
print("VOS CONTRÔLES \n")
display(pd.DataFrame(VERIFICATIONS))

# ── la relecture : des contrôles de FORME, qui ne révèlent aucune valeur.
print("LA RELECTURE \n")
verifier('votre identifiant GitHub est renseigné', GRAINE is not None,
         'les questions 2 et 11 mesurent sur un échantillon tiré de votre identifiant')
verifier('cinq constats', len(CONSTATS) == 5,
         'la question 3 en demande cinq — les trois de la séance plus deux des vôtres')
verifier('au moins deux constats mesurés par vous',
         sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres')
verifier('chaque constat porte un chiffre ET une comparaison',
         all(c['chiffre'] and c['comparaison'] for c in CONSTATS),
         'un constat sans comparaison ne compte pas : « par rapport à quoi ? »')
verifier('trois hypothèses', len(HYPOTHESES) == 3,
         'la question 14 en demande trois')
verifier('chaque hypothèse dit ce qui la contredirait',
         all(str(h['contredite si']).strip() for h in HYPOTHESES),
         'une hypothèse que rien ne pourrait contredire ne vaut rien')
verifier('au moins trois essais à vous — les amorces ne comptent pas',
         sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'la question 8 en demande trois, de vous')
verifier('chaque essai porte un attendu — un signe ET une raison',
         all(e['attendu'].strip() for e in ESSAIS),
         'jamais une valeur : un signe, et pourquoi')
verifier('les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle',
         all(e['prédiction écrite avant ?'] != '— aucune'
             for e in ESSAIS if e['hypothèse'] not in AMORCES),
         'si cette ligne tombe, quelqu un a modifié ESSAIS à la main : noter() le garantit')
verifier('une hypothèse a été testée sous son propre nom',
         any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'la question 14 en teste une : le tableau d essais doit porter son nom')


CONSTATS CHIFFRÉS 



,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucune cible manquante,0 valeur(s) manquante(s),sur 149894 lignes de train,garder la cible telle quelle
4,des ages impossibles sont presents,46 age(s) negatif(s),sur 149894 lignes; zero age negatif est plausible,ecarter la sentinelle negative ou la recoder e...


HYPOTHÈSES DE FEATURES 



""


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 19:45:56"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.49,"oui, à 19:45:56"
2,retirer age,donnee : colonne age retiree,baisse faible ou nulle car les arbres peuvent ...,82.49,"oui, à 19:46:19"
3,learning_rate a 0.05,"reglage : pas d'apprentissage 0,05",score proche ou legerement meilleur avec un pa...,82.54,"oui, à 19:46:19"
4,arbre de decision,famille : arbre de decision,score probablement plus faible qu'un ensemble ...,82.46,"oui, à 19:46:19"
5,indicateur_age_sentinelle,ajout de age_sentinelle via preparer(),le signal existe deja dans age; le nouvel indi...,82.49,"oui, à 19:46:30"


VOS CONTRÔLES 



,contrôle,tient
0,"la part d absents est un pourcentage, et c est...",True
1,"`ma_reference` est un pourcentage, au-dessus d...",True
2,les deux chiffres sont complémentaires,True
3,le prédicteur ne répond que des 0 et des 1,True
4,son score COÏNCIDE avec votre référence,True
...,...,...
70,le signe de la VALEUR apparaît au moins une fois,True
71,"chaque verdict a été choisi, et le tri n est p...",True
72,"chaque colonne signalée est classée, et chaque...",True
73,"la sentinelle est nommée, chiffrée, et jugée",True


LA RELECTURE 

  ok        votre identifiant GitHub est renseigné
  ok        cinq constats
  ok        au moins deux constats mesurés par vous
  ok        chaque constat porte un chiffre ET une comparaison
  À REVOIR  trois hypothèses   ->   la question 14 en demande trois
  ok        chaque hypothèse dit ce qui la contredirait
  ok        au moins trois essais à vous — les amorces ne comptent pas
  ok        chaque essai porte un attendu — un signe ET une raison
  ok        les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle
  À REVOIR  une hypothèse a été testée sous son propre nom   ->   la question 14 en teste une : le tableau d essais doit porter son nom


**Ce qu'on rend, et pourquoi ce format.** Ce qu'on rend n'est pas un carnet : c'est un **texte**.
« Le modèle fait 82 % » ne dit rien ; « la cible est déséquilibrée à 18 %, donc 82 % vaut à peine
mieux que la référence » déclenche une décision. Entre les deux, la différence n'est pas le
chiffre : c'est la comparaison — et c'est exactement ce que les trois tableaux demandent.


## Rendre le travail

1. **Exécutez tout** le carnet (`Run All`), pour que les tableaux soient remplis.
2. **Déposez** ce carnet dans votre dépôt GitHub personnel, **avec** votre déclaration d'usage de
   l'IA, dans le **même commit** :
   - téléchargez le modèle **`DECLARATION_modele.md`** depuis le **dépôt du cours** ;
   - recopiez-le dans votre dépôt sous le nom **`declaration_DM1.md`** ;
   - remplissez-le : les deux scores, ce que vous avez produit seul, ce que vous avez fait
     avec un assistant, et ce que vous ne sauriez pas expliquer.
3. Vérifiez que le lien de votre dépôt est bien à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.
